# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EltunLTN/FlyRank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This notebook frames content decline prediction as a machine learning task. The goal is to help editors decide which content pages should be reviewed first.

> The model is a decision-support tool. It does not automatically change content.

## 1. My lane as an ML task (type)

### Classification

I frame my lane as a **binary classification** problem.

The decision is: **which content pages should an editor review because they may be at risk of declining performance?**

The model would output a probability that a content item is declining. This probability can then be used as a risk score to prioritize pages for editorial review.

Classification is appropriate because the outcome can be represented as two states: **declining** or **not declining**.

The prediction supports an editorial prioritization decision rather than automatically changing a page.

In [1]:
task_type = "binary classification"
model_output = "probability of decline"
decision = "prioritize content pages for editorial review"

print("Task type:", task_type)
print("Model output:", model_output)
print("Decision supported:", decision)


Task type: binary classification
Model output: probability of decline
Decision supported: prioritize content pages for editorial review


## 2. Target or proxy

### Target: content decline

My target is whether a content item declines in performance.

For a production ML system, I would prefer an **observed future outcome**, for example whether a page's impressions decrease by more than a defined threshold during a future time window.

The starter dataset contains `is_declining_label`. This label is derived from `trend_direction`, which is calculated from `trend_pct`. Therefore, this starter label is a **rule-derived teaching target**, not a fully independent future observed outcome.

For this framing exercise, I use `is_declining_label` to understand the target structure. However, `trend_pct` and `trend_direction` must not be used as model features because they directly define the label and would create target leakage.

The intended action is to use the predicted decline risk to help editors decide which pages to inspect first.

In [2]:
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/EltunLTN/FlyRank/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_URL)

print("Dataset shape:", df.shape)
print("Target-related columns:")
print([c for c in df.columns if "declin" in c.lower() or "trend" in c.lower()])

if "is_declining_label" not in df.columns:
    if "trend_direction" not in df.columns:
        raise ValueError("Neither is_declining_label nor trend_direction exists in the dataset.")
    df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
    print("\nis_declining_label was created from trend_direction for this framing exercise.")

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts(dropna=False).sort_index())

print("\nTarget proportions:")
print(df["is_declining_label"].value_counts(normalize=True, dropna=False).sort_index())

print(f"\nDeclining rate: {df['is_declining_label'].mean():.3f}")


Dataset shape: (30000, 44)
Target-related columns:
['trend_direction', 'trend_pct']

is_declining_label was created from trend_direction for this framing exercise.

Target distribution:
is_declining_label
0    13738
1    16262
Name: count, dtype: int64

Target proportions:
is_declining_label
0    0.457933
1    0.542067
Name: proportion, dtype: float64

Declining rate: 0.542


## 3. Success metric

### Precision@K

I would use **Precision@K** as the decision-oriented success metric.

The practical use case is an editorial review queue. Editors have limited time, so they cannot manually inspect every page. The model should place the highest-risk pages near the top of the queue.

Precision@K answers this question:

> Among the K pages the model prioritizes for review, how many are actually declining?

A higher Precision@K means the editorial team spends more of its limited review time on pages that really need attention.

The task itself is classification, but the predicted probability is used to **rank** pages for the editorial queue. Therefore, Precision@K connects the model output directly to the decision.

In [3]:
metric = "Precision@K"
k_description = "Number of highest-risk pages selected for editorial review"

print("Success metric:", metric)
print("K represents:", k_description)
print("Good result: a high proportion of truly declining pages among the top-K predictions.")


Success metric: Precision@K
K represents: Number of highest-risk pages selected for editorial review
Good result: a high proportion of truly declining pages among the top-K predictions.


## 4. The unit of analysis, as a real dataframe

### Unit of analysis: one content page

The unit of analysis is **one content item (page)**.

Each row represents one pseudonymized content page and contains content properties, keyword context, and aggregated performance metrics.

Therefore:

**1 row = 1 content page**

The model would make one prediction for each content page. The resulting risk score could then be used to prioritize pages for editorial review.

In [4]:
preferred_columns = [
    "content_id",
    "client_id",
    "content_type",
    "content_age_days",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position"
]

available_columns = [c for c in preferred_columns if c in df.columns]

if not available_columns:
    raise ValueError("Expected content-level columns were not found in the dataset.")

lane_df = df[available_columns].copy()

print("Unit of analysis: one content page")
print("Rows shown:", min(10, len(lane_df)))
print("Columns shown:", len(lane_df.columns))

display(lane_df.head(10))


Unit of analysis: one content page
Rows shown: 10
Columns shown: 10


,content_id,client_id,content_type,content_age_days,word_count,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position
0,content_304f48230142,client_f369cb89fc,keyword article,187,3221.0,3803,29,17,0.76,10.6
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,2481.0,15320,7,9,0.05,20.3
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,3515.0,12581,11,11,0.09,36.5
3,content_331d6c4de07b,client_19581e27de,keyword article,463,NaN,11751,58,78,0.49,6.2
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,2803.0,19140,24,145,0.13,44.0
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,3080.0,3970,1,5,0.03,8.5
6,content_9a34b442b552,client_8722616204,keyword article,90,3059.0,20,0,1,0.00,7.0
7,content_a63219c6e95a,client_19581e27de,keyword article,445,NaN,1724,1,28,0.06,21.2
8,content_5e6c160719bc,client_6208ef0f77,keyword article,90,3807.0,32574,29,68,0.09,46.0
9,content_c27558df2b0c,client_19581e27de,keyword article,257,NaN,1240,2,3,0.16,4.9


### Target shown against real observations

The following dataframe shows the target alongside several content-level signals. This makes the target concrete rather than describing the unit of analysis only in prose.

In [5]:
target_view_columns = [
    "content_id",
    "content_type",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

available_target_columns = [c for c in target_view_columns if c in df.columns]

display(df[available_target_columns].head(10))


,content_id,content_type,impressions_90d,clicks_90d,sessions_90d,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,keyword article,3803,29,17,down,-41.4,1
1,content_a1fb4e703a9e,keyword article,15320,7,9,down,-57.7,1
2,content_9aa793d4d895,keyword article,12581,11,11,down,-60.9,1
3,content_331d6c4de07b,keyword article,11751,58,78,stable,-13.8,0
4,content_d99b7a2d90ca,keyword article,19140,24,145,down,-34.7,1
5,content_d4084a4bc775,keyword article,3970,1,5,down,-38.9,1
6,content_9a34b442b552,keyword article,20,0,1,down,-92.3,1
7,content_a63219c6e95a,keyword article,1724,1,28,stable,0.6,0
8,content_5e6c160719bc,keyword article,32574,29,68,down,-58.8,1
9,content_c27558df2b0c,keyword article,1240,2,3,down,-29.2,1


## 5. Why ML beats a fixed rule here

A simple rule such as **"review every page with CTR below X"** would be easy to implement, but it would only use one manually chosen threshold.

Content performance depends on multiple interacting signals, including search impressions, clicks, sessions, content age, word count, keyword characteristics, search position, and recent activity.

The relationship between these signals and decline may also differ across content types and clients. A fixed threshold would be difficult to maintain when these patterns change.

ML is useful here because it can combine multiple signals and learn patterns from observed outcomes instead of requiring an editor to manually specify every threshold.

The model would not replace the editor. It would provide a ranked risk signal that helps the editor decide which pages to review first.

### Decision and action

For editors deciding which content pages to review first, I would build a classification model that estimates each page's risk of future decline from historical content and performance signals, evaluate the decision with **Precision@K**, and use the resulting risk score to prioritize editorial review.

A high score does not automatically mean that a page should be changed. It is a **decision-support signal** that helps allocate limited editorial attention.

In [6]:
fixed_rule = "review if CTR < X"
ml_approach = "combine multiple signals and learn patterns"

print("Example fixed rule:", fixed_rule)
print("ML approach:", ml_approach)
print("Supported action: prioritize pages for editorial review")


Example fixed rule: review if CTR < X
ML approach: combine multiple signals and learn patterns
Supported action: prioritize pages for editorial review


## Self-check

- [x] Every section is filled — markdown thinking and code that backs it
- [x] The notebook loads the starter data and shows a real dataframe
- [x] The unit of analysis is one content page
- [x] The task type is clearly identified as binary classification
- [x] The target source and its rule-derived limitation are stated honestly
- [x] Precision@K is connected to the editorial decision
- [x] `trend_pct` and `trend_direction` are identified as leakage and are not proposed as model features
- [x] The output is described as decision-support, not automatic action
- [x] No client names, private queries, or private URLs are used

Before submission, run the notebook top to bottom with **Runtime → Run all**, check that there are no errors, save the executed notebook, commit it under `work/notebooks/`, and submit the repository URL.